← [Overview](../00_overview.ipynb)

# Extremal-prototype selection: k-maxoids

The two previous notebooks — [partitional clustering](01_partitional_clustering.ipynb) and
[agglomerative clustering](02_agglomerative_clustering.ipynb) — chase the **same goal**:
representatives that sit in the *middle* of their groups, minimising the within-cluster
distance $J$.

**k-maxoids does the opposite.** It selects the $k$ periods that are **as far apart as
possible**, so the representatives reach the *extremes* of the data and their convex hull
approximates the convex hull of the whole dataset. This is a different paradigm —
**extremal-prototype selection**, also known as *archetypal analysis*, introduced by
[Sifa & Bauckhage (2017)](https://doi.org/10.1109/DSAA.2017.76).

Where it helps: capacity-expansion models that must stay **feasible** under the worst period
care about extremes, not averages — exactly what k-maxoids targets.

---

## The algorithm end to end

This notebook follows a single run of k-maxoids from input to output:

| Stage | What it is | Detail |
|---|---|---|
| **In** | the period matrix $D$ (+ a number $k$) | $N_p$ periods, each a point in $N_a \cdot N_t$-dimensional feature space; $k$ is how many representatives to keep |
| **Transform** | a greedy **swap loop**, repeated | start from $k$ random periods, repeatedly swap in periods that push the representatives further apart, then restart many times |
| **Out** | $k$ maxoids + one label per period | $k$ **real** periods (never synthetic averages) and, for every period, the nearest maxoid it is attached to |

The sections below open each stage in turn.

---

## 1  What comes in: the period matrix

Clustering starts from the output of [preprocessing](../01_preprocessing.ipynb): each attribute
normalised to $[0, 1]$ and the flat series **unstacked** so every one of the six days becomes a
single row-vector. Each row is one period — a point in the $N_a \cdot N_t = 2 \times 4 = 8$-dimensional
feature space. k-maxoids will keep $k$ of these six points as representatives.

In [ ]:
import numpy as np
import pandas as pd

import tsam
from tsam import ClusterConfig

# Preprocessed period matrix D (normalised + unstacked) from 01_preprocessing.
tiny_period_df = pd.read_csv(
    "../../../data/tiny_periods.csv", header=[0, 1], index_col=0
)
input_array = tiny_period_df.values  # shape (6, 8): six periods, eight features
N_PERIODS = input_array.shape[0]


print("period matrix D:", input_array.shape)
tiny_period_df.round(4)

---

## 2  The objective: push the representatives apart

k-maxoids measures a set of representatives $M = \{M_1, \dots, M_k\}$ by their **total spread** —
the sum of squared distances over all *pairs* of representatives:

$$
E(M) = \sum_{i < j} \lVert M_i - M_j \rVert^2
$$

The algorithm drives $E$ **up**: the further apart the representatives, the better.

Two more quantities fall out of any representative set and complete the picture — we will need
all three below:

* `spread_E(M)` — the objective $E$ the swap loop maximises;
* `assign(M)` — attaches every period to its nearest representative; this partition is part of
  the **output**;
* `inertia(M)` — the total period-to-representative distance; section 4 uses it to choose between
  random restarts.

In [ ]:
def spread_E(M):
    """Total spread of a representative set: sum of squared distances over all PAIRS.

    M : np.ndarray, shape (k, N_a * N_t) — the k representatives, one per row.

    This is the objective the k-maxoids swap loop drives UP.
    """
    k = len(M)
    return float(
        sum(np.sum((M[i] - M[j]) ** 2) for i in range(k) for j in range(i + 1, k))
    )


def _dists(M):
    """(N_PERIODS, k) matrix of distances from every period to every representative."""
    return np.sqrt(
        ((input_array[:, None, :] - np.asarray(M)[None, :, :]) ** 2).sum(axis=-1)
    )


def assign(M):
    """Attach every period to its nearest representative — the partition k-maxoids returns."""
    return np.argmin(_dists(M), axis=1).tolist()


def inertia(M):
    """Total distance from every period to its NEAREST representative.

    Section 4 scores the 100 random restarts with this and keeps the smallest.
    """
    return float(np.min(_dists(M), axis=1).sum())

---

## 3  How the data is transformed: the greedy swap loop

One run of k-maxoids maximises $E$ with a cheap greedy loop. The representatives start as $k$
real periods; then, sweeping over every period $x$ in turn:

1. Find the slot $a$ of the representative $x$ is **nearest** to:
   $\;a = \arg\min_i \lVert M_i - x \rVert^2$.
2. Would putting $x$ in that slot spread the representatives further apart? Compare $x$ against
   the *other* representatives with the incumbent $M_a$ against those same others:

$$
\underbrace{\sum_{i \neq a} \lVert x - M_i \rVert^2}_{\text{spread with } x}
\;>\;
\underbrace{\sum_{i \neq a} \lVert M_a - M_i \rVert^2}_{\text{spread of current representative}}
\quad\Longrightarrow\quad M_a \leftarrow x .
$$

3. Swap only if the inequality is **strict**; ties keep the incumbent.

Each accepted swap raises $E$, so a sweep never makes the set worse. tsam's implementation
(`k_maxoids.py`) runs a **fixed** number of sweeps per run (`n_passes = 5`). Once no swap helps, further sweeps simply change nothing.

### Tracing one run

A trace of **one** k-maxoids run on the six periods ($k = 3$), mirroring
`k_maxoids.py`. tsam seeds the starting representatives at random; here they are fixed at days
0, 1 and 2 so the trace is reproducible.

In [ ]:
n_passes = 5
representative_indices = [
    0,
    1,
    2,
]  # initial representatives (tsam picks these at random)
representative_periods = input_array[representative_indices].copy()
print(
    f"initial representatives: {representative_indices}  (E = {spread_E(representative_periods):.4f})"
)

for sweep in range(n_passes):
    swapped = []
    for current_period_index in range(N_PERIODS):
        current_period = input_array[current_period_index]
        # squared distance from the current period to each representative
        squared_distances = np.sum(
            (representative_periods - current_period) ** 2, axis=1
        )
        nearest_representative_index = int(
            np.argmin(squared_distances)
        )  # representative the period is NEAREST to
        # spread if we swapped this period in: the period vs the OTHER representatives
        spread_with_candidate = (
            squared_distances.sum() - squared_distances[nearest_representative_index]
        )
        # spread of the incumbent: the current representative vs the OTHER representatives
        incumbent_squared_distances = np.sum(
            (
                representative_periods
                - representative_periods[nearest_representative_index]
            )
            ** 2,
            axis=1,
        )
        spread_of_incumbent = (
            incumbent_squared_distances.sum()
            - incumbent_squared_distances[nearest_representative_index]
        )
        if (
            spread_with_candidate > spread_of_incumbent
        ):  # swapping pushes the representatives further apart
            representative_periods[nearest_representative_index] = current_period
            representative_indices[nearest_representative_index] = current_period_index
            swapped.append(
                f"day_{current_period_index}->slot{nearest_representative_index}"
            )
    print(
        f"  sweep {sweep + 1}: {(', '.join(swapped) if swapped else 'no swaps'):24s}"
        f" representatives now {representative_indices}  (E = {spread_E(representative_periods):.4f})"
    )

print(
    f"\nfinal representatives of THIS run: {representative_indices}  (E = {spread_E(representative_periods):.4f})"
)
print(
    f"  its partition (each day -> nearest representative): {assign(representative_periods)}"
)
print(f"  its inertia = {inertia(representative_periods):.4f}")

The swap loop settles after a single sweep on representatives **`[0, 1, 5]`** and then stops
changing. Look at what it kept: **two near-identical sunny days** (0 and 1) as *separate*
representatives. That is not a mistake — `[0, 1, 5]` genuinely is the spread-maximal triple (the
next section checks all 20 of them). It is what maximising $E$ *means*: the objective rewards
reaching the extremes and is perfectly happy to spend two of its three representatives on the
**same** extreme, leaving the overcast middle of the data with no representative of its own.

The cost lands in the **partition**: with no representative near days 2–3, `assign` splits the
sunny pair across two clusters (`[0, 1, 0, 1, 2, 2]`). A high $E$ does not guarantee a sensible
partition — which is why a single run is not the whole algorithm.

---

## 4  From one run to the answer: 100 restarts, keep the best

A single swap loop can land on the degenerate `[0, 1, 5]` above. k-maxoids guards against that by
running the whole loop **100 times** (`n_init = 100`), each from a fresh random start, and keeping
just one. The tie-break is the key design choice: runs are scored **not** by the spread $E$ they
maximised, but by **inertia** — the total period-to-representative distance. The run whose
partition has the **lowest** inertia wins.

With six periods and $k = 3$ there are only $\binom{6}{3} = 20$ possible representative sets, so we
can score **all** of them and see exactly what the restart-and-keep-best rule chooses between.

In [ ]:
from itertools import combinations

table = pd.DataFrame(
    [
        {
            "representatives": str(list(triple)),
            "E (spread)": spread_E(input_array[list(triple)]),
            "inertia": inertia(input_array[list(triple)]),
            "partition": str(assign(input_array[list(triple)])),
        }
        for triple in combinations(range(N_PERIODS), 3)
    ]
)

print("Highest SPREAD  — what a single swap loop chases (maximise E):")
print(table.nlargest(3, "E (spread)").round(4).to_string(index=False))

print("\nLowest INERTIA  — what the 100-restart tie-break keeps:")
print(table.nsmallest(3, "inertia").round(4).to_string(index=False))

The two rules point at **different** representative sets, with **different** partitions:

* The **spread-maximal** set is `[0, 1, 5]` from the trace — the degenerate one. Its partition
  `[0, 1, 0, 1, 2, 2]` tears the sunny pair apart, and its inertia is more than **double** the
  best available.
* The **inertia-minimal** sets all reach the partition `[0, 0, 1, 1, 2, 2]` — the six days grouped
  into their three natural shape-pairs (sunny / overcast / cloudy), one representative per pair.

So the swap loop *explores* extreme representative sets, and the inertia tie-break then *discards*
the ones that partition the data badly. What k-maxoids returns is whichever high-spread set also
gives the lowest-inertia partition.

---

## 5  What comes out: running k-maxoids in tsam

In tsam the whole procedure is one call. The method is `kmaxoids`; its representation defaults to
`maxoid`, so the profile reported for each cluster is the actual extreme period selected above —
not an average.

In [ ]:
# k-maxoids: selects the k most mutually dissimilar periods.
# representation defaults to 'maxoid' for kmaxoids.
cfg_kmaxoids = ClusterConfig(method="kmaxoids", representation="maxoid")
print(cfg_kmaxoids)

In [ ]:
# k_maxoids draws its 100 random restarts from numpy's GLOBAL RNG and takes no seed
# argument, so seed it here to make the output labels below reproducible.
np.random.seed(0)
# Raw tiny series — for the tsam.aggregate call in section 5 (it normalises internally).
tiny = pd.read_csv("../../../data/tiny.csv", index_col=0, parse_dates=True)
result_kmx = tsam.aggregate(
    tiny, n_clusters=3, period_duration="1D", cluster=ClusterConfig(method="kmaxoids")
)
print(
    "k-maxoids assignments:",
    np.asarray(result_kmx.cluster_assignments),
    "  weighted RMSE:",
    round(result_kmx.accuracy.weighted_rmse, 4),
)

**Reading the output.** k-maxoids returns two things, both visible here:

* **the representatives** — three real periods (the maxoids), reported as the profile of each
  cluster; and
* **`cluster_assignments`** — one entry per period, in period order: entry `p` is the cluster that
  **day `p`** landed in. So `[2 2 1 1 0 0]` reads days 0–1 → cluster 2, days 2–3 → cluster 1,
  days 4–5 → cluster 0.

The cluster **labels** (which group is called 0, 1 or 2) depend on the winning restart, so without
the seed above they can change from run to run. The **partition**, though, is stable: the three
shape-pairs — exactly what section 4 said the inertia tie-break would keep.

On this tiny, well-separated dataset the extremes and the natural groups happen to line up. Do not
read that as the general case: on larger data, where the modelling risk sits in the peaks, the
spread objective is precisely what pulls representatives out to those peaks — the reason to reach
for k-maxoids in the first place.

---

## Aside: two things called "maxoid"

Because the same word names both an algorithm and a representation, they are easy to conflate — but
they are **different steps of the pipeline**:

| | k-maxoids **clustering** (this notebook) | maxoid **representation** ([Representation](../03_representation.ipynb)) |
|---|---|---|
| Pipeline step | *grouping* — which periods form a cluster | *representation* — which profile stands for a formed cluster |
| What it does | selects $k$ spread-maximal periods, then assigns nearest members | picks the most extreme member of an **already-formed** cluster |
| Works with | its own maxoid centres | **any** clustering (e.g. Ward + maxoid) |

Both target **extremes**, which also links them to [Extreme periods](../04_extreme_periods.ipynb) —
the explicit way to force peak/trough periods to survive aggregation. k-maxoids reaches for
extremes *automatically*; extreme-period handling does it *by construction*.

---

**Up next:**
* [Averaging](04_averaging.ipynb) — the one time-based grouping method
* [Representation](../03_representation.ipynb) — the maxoid representation and its alternatives
* [Extreme periods](../04_extreme_periods.ipynb) — forcing peaks to survive aggregation
* [Comparing clustering methods](../../../tutorials/comparing_clustering_methods.ipynb) — all methods side by side on the same data